# Rewrite solve_mix functions

In [1]:
import sys
sys.path.append("../") 

In [2]:
import cvxpy as cp

In [6]:
from flow.solve_mix import *
from flow.interface import *

In [ ]:
def med_plus_problem(
    problem_spec: ProblemSpec,
    prior_prob: list[float] | None = None,
) -> cp.Problem:
    """MED+ is minimum error discrimination that includes an additional
    POVM element that corresponds to inconclusive outcomes.
    """

    assert problem_spec.state_type == "densitymatrix"
    logger = logging.getLogger(__name__)

    k = problem_spec.num_states

    # PI is the variable for the POVM elements we try to solve for.
    PI_list = [
        cp.Variable(
            shape=(problem_spec.num_amps, problem_spec.num_amps),
            hermitian=True,
            name=f"PI_{i}",
        )
        for i in range(k + 1)
    ]

    # https://www.cvxpy.org/tutorial/dpp/index.html
    # DPP forbids taking the product of two parametrized expressions,
    # so we will not parametrize prior probabilities.
    # Actually for this formulation, it is fine to parametrize it.
    # For code uniformity, we will not do it.
    # prior_prob = cp.Parameter(k, name="prior_prob")
    if prior_prob is None:
        prior_prob = np.ones(k) * (1 / k)
        logger.info(f"The prior probabilities is set to uniform (k = {k})")
    else:
        logger.info(f"The prior probabilities is set to {prior_prob}")

    prob_succ_expr = get_prob_succ_expr(
        problem_spec=problem_spec, prior_prob=prior_prob, PI_list=PI_list
    )

    objective = cp.Maximize(prob_succ_expr)

    constraints = []

    # Constraint 1. Positive operators
    for i in range(k + 1):
        constraints.append(PI_list[i] >> 0)

    # Constraint 2. Completeness
    I = np.identity(problem_spec.num_amps)
    constraints.append(cp.sum(PI_list) == I)

    return cp.Problem(objective, constraints)


In [7]:
def crossQSD_problem(
    problem_spec: ProblemSpec,
    prior_prob: list[float] | None = None,
    alpha: list[float] | None = None,
    beta: list[float] | None = None,
) -> cp.Problem:
    objective = 0
    constraints = 0
    return cp.Problem(objective, constraints)